
### Learning delta storage cost optimization via vacuum

通过上一节的“时间旅行”，已经感受到了 Delta Lake 多版本共存的逆天威力。但天下没有免费的午餐，那些被标记为“已过期”的历史 Parquet 文件，依然雷打不动地躺在云端硬盘（如AWS S3 / Azure Blob）上，日复一日常年累月地消耗着公司的真金白银。

如果不加控制，一张原本只有 100GB 数据的资产表，由于每天频繁的增量 Upsert 产生的大量历史碎片文件，在物理层面上会迅速滚雪球到几个 TB，直接让公司账单彻底原地爆炸！

VACUUM（真空清理）就是 Delta Lake 的“物理清洁工”与“成本刹车阀”。它的职责是强行越过元数据账本，直接进入物理存储层，将那些超过保留期限的历史死文件彻底擦除，在物理硬盘上“毁尸灭迹”来释放宝贵的存储空间。


**但这是一把极其危险的“双刃剑”：一旦文件被 VACUUM 物理粉碎，对应版本的时间旅行超能力将永久失效。因此，如何安全设置保留时间、严防误删可用版本，是每一个高阶架构师的必修课**



In [0]:
table_detail = spark.sql("DESCRIBE DETAIL gold_seller_dimension").select("location", "numFiles").collect()[0]
print(f"📄 当前账本认为的‘最新活跃’文件数: {table_detail['numFiles']} 个")


In [0]:
# 查看这张表的全部前世今生链条
print("账本中的历史快照链条：")
spark.sql("DESCRIBE HISTORY gold_seller_dimension").select("version", "operation", "timestamp").show()

在大厂标准中，Delta Lake 默认开启了一道安全死线：禁止清空 7 天（168小时）以内的任何历史文件，严防开发人员不小心把最近几小时内别人正在分析或回溯的可用版本一巴掌拍死。

我们在下面的实验中，只留24小时内的数据

In [0]:
spark.sql("VACUUM gold_seller_dimension LITE RETAIN 24 HOURS")

In [0]:
try:
    print("尝试回溯Version1: ...")
    df_v1 = spark.read.format("delta").option("versionAsOf", 1).load("gold_seller_dimension")
    df_v1.show()
except Exception as e:
    print(f"👉 {str(e)[:150]}...")
    print("\n✅ 对账结论：老文件已被彻底粉碎，账本中虽然还记着 Version 1 这个名字，但底层的物理肉身已经没了，所以时间旅行成功报错拦截！")

#### 验证当前主线正常读写:
老底座被抽空了，那么当前最新的生产表还能正常提供服务吗？我们立刻执行一次标准的增量读写，测试一下

In [0]:
print("读取当前主时间线数据: ")
spark.table("gold_seller_dimension").show()


测试写入数据是否正常

In [0]:
test_columns = ["seller_id", "seller_name", "total_sales", "last_update_date"]
test_data = [(999, "Vacuum Safe Shop", 8888.0, "2026-06-16")]

df_test = spark.createDataFrame(test_data,test_columns)
df_test.write.mode("append").saveAsTable("gold_seller_dimension")

spark.table("gold_seller_dimension").show()

测试成功：删除时间线后，仍然读写正常